# 🕐📅🕓📅 <span style="color: white; background-color: Teal"><b> Carga do Proceso de Período de Experiência </b></span></p>

🧩 <span style="color: MediumSeaGreen"><b>  1- Carrega e valida todas as bases necessárias </b></span></p>
- O script abre o arquivo PROCESSOS.xlsx para registrar cada etapa executada  
- Base de colaboradores  
- Base de afastamentos (exceto férias)  
- Base de unidades
- Tudo é carregado via pandas, openpyxl e caminhos de rede fixos

📅 <span style="color: MediumSeaGreen"><b> 2- Gera um calendário completo por colaborador </b></span></p>
- Cria uma linha do tempo diária entre a admissão e o fim do ano  
- Expande registros unindo colaboradores × dias trabalháveis  
- Junta informações de ausência por dia  
- Marca cada dia como “trabalhado” ou “afastado”
- Isso permite um cálculo dia a dia, não apenas baseado em datas calendário

🩺 <span style="color: MediumSeaGreen"><b> 3- Identifica ausências (exceto férias) e calcula impacto no período de experiência </b></span></p>
- Marca cada dia de afastamento  
- Calcula total de dias afastados  
- Calcula se o colaborador estava afastado em cada intervalo crítico  
- Recalcula contagem de dias úteis considerando ausências
- Isso garante que a previsão dos 45 e 90 dias seja realista e justa, respeitando legislação e práticas internas

🎯 <span style="color: MediumSeaGreen"><b> 4- Calcula previsão de 45 dias e 90 dias </b></span></p>
- A lógica central do script para cada colaborador
    - Soma dias trabalhados até chegar a 45
    - Ignora dias afastados
    - Registra a data exata em que os 45 dias são completados
    - Repete o processo para 90 dias
- Além disso
    - Marca se houve afastamento até os 45 dias  
    - Marca se houve afastamento até os 90 dias  
    - Marca se está afastado atualmente
- O resultado é altamente útil para prever término de contrato de experiência e programar avaliações

📊 <span style="color: MediumSeaGreen"><b> 5- Consolida tudo em uma base final </b></span></p>
- Registro  
- Previsao_45  
- Afastou_ate_45  
- Previsao_90  
- Afastou_ate_90  
- Afastado_atualmente  
- Total_dias_afastados
- Ou seja, um painel completo por colaborador

📁 <span style="color: MediumSeaGreen"><b> 6- Gera o Excel estruturado para uso corporativo </b></span></p>
- O script exporta um arquivo final em Excel
    - Com tabela formatada  
    - Estilo TableStyleLight13  
    - Pronto para consumo pelo RH 

📈 <span style="color: MediumSeaGreen"><b> 7- Atualiza automaticamente o Power BI — Período de Experiência </b></span></p>
- Abre o PBIX  
- Dá refresh  
- Publica  
- Fecha o Power BI
- Tudo via pyautogui, simulando uso humano, mas com execução 100% automatizada

🧮 <span style="color: MediumSeaGreen"><b> 8- Registra todas as etapas no controle de processos </b></span></p>
- ID do processo  
- Timestamp  
- Número da etapa
- No PROCESSOS.xlsx
- Isso garante rastreabilidade e auditoria

⏱️ <span style="color: MediumSeaGreen"><b> 9- Exibe o resumo da execução </b></span></p>
- Status final  
- Tempo total de execução  
- Carimbo de hora inicial e final

# Importação das Bibliotecas

In [3]:
import os
import pandas as pd
import pyautogui
import pyperclip
import time
from datetime import datetime, date
from openpyxl import Workbook, load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Processo de Importação finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Processo de Importação finalizado

----------------------------------------------------------------------------------------------------


# Carregando Base de Controle de Processos

In [4]:
# Carregamento da base de controle de processos
id = 1

path_registros_processos = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'
registros_processos = pd.read_excel(path_registros_processos, sheet_name = "REGISTROS", engine='openpyxl')
wb_p = load_workbook(path_registros_processos)
ws_p = wb_p['REGISTROS']

# Controle de atualização de processo: Etapa 0
tempo_0 = [id, datetime.today(), 0]
ws_p.append(tempo_0)
wb_p.save(path_registros_processos)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Registro de processos')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Registro de processos

----------------------------------------------------------------------------------------------------


# Carregamento de Bases

In [5]:
# Base de colaboradores
colaboradores = pd.read_excel(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\COLABORADORES.xlsx')

# Base de afastamentos
afastamentos = pd.read_excel(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\AFASTAMENTOS.xlsx')

# Base de unidades
path_unidades = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\DIMENSÕES POWER BI.xlsx'
unidades = pd.read_excel(path_unidades, sheet_name='UNIDADES', usecols='B:F', skiprows=3)

# Controle de atualização de processo: Etapa 1
tempo_1 = [id, datetime.today(), 1]
ws_p.append(tempo_1)
wb_p.save(path_registros_processos)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Carregamento de bases concluído')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Carregamento de bases concluído

----------------------------------------------------------------------------------------------------


# Inicializando as Variáveis

In [6]:
# Filtrando apenas os colaboradores que foram admitidos a partir de 2024
colaboradores['data_admissao'] = pd.to_datetime(colaboradores['data_admissao'])
colaboradores = colaboradores[colaboradores['data_admissao'] >= '2024-01-01'].copy()

# Base de afastamentos excluindo afastamentos por férias
filtro_ferias = 'Gozo de férias ou recesso - Afastamento temporário para o gozo de férias ou recesso'
afastamentos_sem_ferias = afastamentos[afastamentos['descricao_afastamento'] != filtro_ferias].copy()
afastamentos_sem_ferias['data_afastamento'] = pd.to_datetime(afastamentos_sem_ferias['data_afastamento'])
afastamentos_sem_ferias['data_retorno'] = pd.to_datetime(afastamentos_sem_ferias['data_retorno'])

# Variável com a data de afastamento mais antiga 
data_inicio_calendar = afastamentos_sem_ferias['data_afastamento'].min()

# Variável com a maior data de retorno de afastamento 
hoje = pd.Timestamp(datetime.today().date())
max_retorno = afastamentos_sem_ferias['data_retorno'].max()

if pd.isna(max_retorno) or max_retorno < hoje:
    base_date = hoje
else:
    base_date = max_retorno
    
data_fim_calendar = base_date + pd.Timedelta(days=90)

# Controle de atualização de processo: Etapa 2
tempo_2 = [id, datetime.today(), 2]
ws_p.append(tempo_2)
wb_p.save(path_registros_processos)

print('\n\n   ✅ Variáveis definidas\n\n')



   ✅ Variáveis definidas




# Criação da Tabela de Dias e Registros

In [7]:
# Criação da tabela calendário
calendario = pd.DataFrame({'Data': pd.date_range(start=data_inicio_calendar, end=data_fim_calendar, freq='D')})

# Base com os registros únicos de colaboradores
registros_unicos = colaboradores[['registro']].drop_duplicates().rename(columns={'registro': 'Registro'})

# Base com todas as datas e todos os registros
registro_vs_datas = registros_unicos.merge(calendario, how='cross')

# Base com todas as datas e todos os registros, filtrando datas após admissão de cada colaborador
colab_admissao = colaboradores[['registro', 'data_admissao']].rename(columns={'registro': 'Registro'})
registro_vs_datas_consolidado = registro_vs_datas.merge(colab_admissao, on='Registro', how='inner')
registro_vs_datas_consolidado = registro_vs_datas_consolidado[registro_vs_datas_consolidado['Data'] >= registro_vs_datas_consolidado['data_admissao']]
registro_vs_datas_consolidado = registro_vs_datas_consolidado[['Registro', 'Data']].copy()

# Controle de atualização de processo: Etapa 3
tempo_3 = [id, datetime.today(), 3]
ws_p.append(tempo_3)
wb_p.save(path_registros_processos)

print('\n\n   ✅ Tabelas criadas\n\n')



   ✅ Tabelas criadas




# Base com Marcação de Afastamento e Contagem de Dias

In [8]:
# Prepara base de afastamentos
afast = afastamentos_sem_ferias.copy()

# Padroniza o nome da coluna de registro (corrige problemas de maiúsculas/minúsculas)
afast.columns = ['Registro' if col.upper() == 'REGISTRO' else col for col in afast.columns]

# Garante que as colunas de data são datetime
afast['data_afastamento'] = pd.to_datetime(afast['data_afastamento'])
afast['data_retorno'] = pd.to_datetime(afast['data_retorno'])

# Expande os períodos de afastamento em dias individuais (Evita MemoryError)
linhas_afastados = []
for _, row in afast.iterrows():
    if pd.notna(row['data_afastamento']):
        # Se não tem data de retorno, consideramos até o fim do calendário
        fim = row['data_retorno'] if pd.notna(row['data_retorno']) else data_fim_calendar
        
        # Garante que não vamos expandir datas infinitas (ex: ano 2050)
        if fim > data_fim_calendar:
            fim = data_fim_calendar
            
        # Cria uma linha para cada dia de afastamento do colaborador
        if row['data_afastamento'] <= fim:
            datas = pd.date_range(start=row['data_afastamento'], end=fim)
            df_temp = pd.DataFrame({'Registro': row['Registro'], 'Data': datas})
            linhas_afastados.append(df_temp)

# Concatena todos os dias de afastamento
if linhas_afastados:
    afastados_dates = pd.concat(linhas_afastados, ignore_index=True).drop_duplicates()
    afastados_dates['Afastado'] = 1
else:
    afastados_dates = pd.DataFrame(columns=['Registro', 'Data', 'Afastado'])

# Base com todos os registros e datas tendo flag se esteve afastado
afastados_dia = registro_vs_datas_consolidado.merge(afastados_dates, on=['Registro', 'Data'], how='left')
afastados_dia['Afastado'] = afastados_dia['Afastado'].fillna(0).astype(int)
afastados_dia['Apto'] = 1 - afastados_dia['Afastado']

# Ordenação necessária para as funções de janela (window functions)
afastados_dia = afastados_dia.sort_values(['Registro', 'Data']).reset_index(drop=True)

# Base com a criação do campo que agrupa momentos de afastamentos (soma acumulada de dias aptos)
afastados_dia['Grupo'] = afastados_dia.groupby('Registro')['Apto'].cumsum()

# Criação da contagem de dias acumulados e zerando quando apto ao trabalho
afastados_dia['Dias_afastados'] = afastados_dia.groupby(['Registro', 'Grupo'])['Afastado'].cumsum()

# Base com apenas os colaboradores que tiveram afastamentos acima de 15 dias
max_dias = afastados_dia.groupby(['Registro', 'Grupo'])['Dias_afastados'].max().reset_index()
grupos_maior_15 = max_dias[max_dias['Dias_afastados'] > 15][['Registro', 'Grupo']].copy()
grupos_maior_15['Contabiliza_Flag'] = 0

# Base consolidada de colaboradores e datas com marcação se contabiliza para período de experiência
contabiliza_periodo_experiencia = afastados_dia.merge(grupos_maior_15, on=['Registro', 'Grupo'], how='left')
contabiliza_periodo_experiencia['Contabiliza'] = contabiliza_periodo_experiencia['Contabiliza_Flag'].fillna(1).astype(int)
contabiliza_periodo_experiencia = contabiliza_periodo_experiencia.drop(columns=['Contabiliza_Flag'])

# Cálculos de totais e validadores
contabiliza_periodo_experiencia['Total_dias'] = contabiliza_periodo_experiencia.groupby('Registro')['Contabiliza'].cumsum()
contabiliza_periodo_experiencia['Validador'] = contabiliza_periodo_experiencia.groupby('Registro').cumcount() + 1
contabiliza_periodo_experiencia['Total_dias_afastados'] = contabiliza_periodo_experiencia['Validador'] - contabiliza_periodo_experiencia['Total_dias']

# Controle de atualização de processo: Etapa 4
tempo_4 = [id, datetime.today(), 4]
ws_p.append(tempo_4)
wb_p.save(path_registros_processos)

print('\n\n   ✅ Marcação de Afastamento e Contagem de Dias concluído\n\n')



   ✅ Marcação de Afastamento e Contagem de Dias concluído




# Bases com Marcação da Data de 45 Dias

In [9]:
# Base com os colaboradores únicos
registros_distintos = contabiliza_periodo_experiencia[['Registro']].drop_duplicates()

# Base com os colaboradores que alcançaram ou alcançarão os 45 dias de experiência
Finalizaram_45_dias = contabiliza_periodo_experiencia[contabiliza_periodo_experiencia['Total_dias'] == 45].copy()
Finalizaram_45_dias = Finalizaram_45_dias.sort_values('Data').drop_duplicates(subset=['Registro'], keep='first')
Finalizaram_45_dias['Teve_afastamento'] = (Finalizaram_45_dias['Total_dias'] != Finalizaram_45_dias['Validador']).astype(int)
Finalizaram_45_dias = Finalizaram_45_dias[['Registro', 'Data', 'Teve_afastamento']]

# Base com os colaboradores que não alcançaram ou alcançarão os 45 dias de experiência considerando afastamentos
nao_localizados = registros_distintos[~registros_distintos['Registro'].isin(Finalizaram_45_dias['Registro'])]

atual_45 = contabiliza_periodo_experiencia[(contabiliza_periodo_experiencia['Registro'].isin(nao_localizados['Registro'])) & (contabiliza_periodo_experiencia['Data'] == hoje)].copy()
atual_45['Data'] = atual_45['Data'] + pd.to_timedelta(45 - atual_45['Total_dias'], unit='D')
atual_45['Teve_afastamento'] = 1
Nao_finalizaram_45_dias = atual_45[['Registro', 'Data', 'Teve_afastamento']]

# Base consolidada de colaboradores e previsão de quando fizeram ou farão 45 dias de experiência
Base_45_dias_consolidado = pd.concat([Finalizaram_45_dias, Nao_finalizaram_45_dias], ignore_index=True)

# Controle de atualização de processo: Etapa 5
tempo_5 = [id, datetime.today(), 5]
ws_p.append(tempo_5)
wb_p.save(path_registros_processos)

print('\n\n   ✅ Marcação da data de 45 dias concluído\n\n')



   ✅ Marcação da data de 45 dias concluído




# Bases com Marcação da Data de 90 Dias

In [10]:
# Base com os colaboradores que alcançaram ou alcançarão os 90 dias de experiência
Finalizaram_90_dias = contabiliza_periodo_experiencia[contabiliza_periodo_experiencia['Total_dias'] == 90].copy()
Finalizaram_90_dias = Finalizaram_90_dias.sort_values('Data').drop_duplicates(subset=['Registro'], keep='first')
Finalizaram_90_dias['Teve_afastamento'] = (Finalizaram_90_dias['Total_dias'] != Finalizaram_90_dias['Validador']).astype(int)
Finalizaram_90_dias = Finalizaram_90_dias[['Registro', 'Data', 'Teve_afastamento']]

# Base com os colaboradores que não alcançaram ou alcançarão os 90 dias de experiência considerando afastamentos
nao_localizados_90 = registros_distintos[~registros_distintos['Registro'].isin(Finalizaram_90_dias['Registro'])]

atual_90 = contabiliza_periodo_experiencia[(contabiliza_periodo_experiencia['Registro'].isin(nao_localizados_90['Registro'])) & (contabiliza_periodo_experiencia['Data'] == hoje)].copy()
atual_90['Data'] = atual_90['Data'] + pd.to_timedelta(90 - atual_90['Total_dias'], unit='D')
atual_90['Teve_afastamento'] = 1
Nao_finalizaram_90_dias = atual_90[['Registro', 'Data', 'Teve_afastamento']]

# Base consolidada de colaboradores e previsão de quando fizeram ou farão 90 dias de experiência
Base_90_dias_consolidado = pd.concat([Finalizaram_90_dias, Nao_finalizaram_90_dias], ignore_index=True)

# Controle de atualização de processo: Etapa 6
tempo_6 = [id, datetime.today(), 6]
ws_p.append(tempo_6)
wb_p.save(path_registros_processos)

print('\n\n   ✅ Marcação da data de 90 dias concluído\n\n')



   ✅ Marcação da data de 90 dias concluído




# Base Final

In [11]:
base_hoje = contabiliza_periodo_experiencia[contabiliza_periodo_experiencia['Data'] == hoje].copy()

# Renomeando colunas antes do merge para manter a estrutura limpa
b45 = Base_45_dias_consolidado.rename(columns={'Data': 'Previsao_45', 'Teve_afastamento': 'Afastou_ate_45'})
b90 = Base_90_dias_consolidado.rename(columns={'Data': 'Previsao_90', 'Teve_afastamento': 'Afastou_ate_90'})

# Merges
base_final = base_hoje.merge(b45, on='Registro', how='left')
base_final = base_final.merge(b90, on='Registro', how='left')

# Criando a flag de afastado atualmente
base_final['Afastado_atualmente'] = (base_final['Apto'] == 0).astype(int)

# Selecionando colunas finais
base_final = base_final[['Registro', 'Previsao_45', 'Afastou_ate_45', 'Previsao_90', 'Afastou_ate_90', 'Afastado_atualmente', 'Total_dias_afastados']]

# Tratamento de valores nulos (NaT) para não dar erro no OpenPyXL ao exportar
base_final = base_final.replace({pd.NaT: None})

# Controle de atualização de processo: Etapa 7
tempo_7 = [id, datetime.today(), 7]
ws_p.append(tempo_7)
wb_p.save(path_registros_processos)

print('\n\n   ✅ Base final realizada\n\n')



   ✅ Base final realizada




# Geração do Arquivo

In [12]:
# Diretório onde será salvo o arquivo final
caminho_arquivo_final = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PERIODO DE EXPERIÊNCIA.xlsx'

# Gerando o arquivo
wb = Workbook()
ws = wb.active
ws.title = "EXPERIENCIA"

for r in dataframe_to_rows(base_final, index=False, header=True):
    ws.append(r)

tabela_base_final = Table(displayName="EXPERIENCIA", ref=ws.dimensions)

estilo_tabela = TableStyleInfo(
    name="TableStyleLight13", 
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=True
)
tabela_base_final.tableStyleInfo = estilo_tabela
ws.add_table(tabela_base_final)
wb.save(caminho_arquivo_final)

# Controle de atualização de processo: Etapa 8
tempo_8 = [id, datetime.today(), 8]
ws_p.append(tempo_8)
wb_p.save(path_registros_processos)

# Atualização do Power BI - PERÍODO DE EXPERIÊNCIA

In [13]:
pyautogui.PAUSE = 1

# Entrar no Power BI
path_pbi = r"X:\Gestão de Pessoas\Analytics\09 - Power BI\PERÍODO DE EXPERIÊNCIA.pbix"
os.startfile(path_pbi) # Abre o arquivo
time.sleep(65)

# Atualizar Power BI
pyautogui.click(x=753, y=103) # Clicar em Atualizar
time.sleep(60)
pyautogui.click(x=1293, y=98) # Publicar
time.sleep(5)
pyautogui.click(x=863, y=477) # Salvar
time.sleep(5)
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(1)
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(7)
pyautogui.click(x=871, y=577) # Substituir
time.sleep(10)
pyautogui.click(x=990, y=585) # Clicar em Entendi
time.sleep(3)
pyautogui.hotkey("Alt", "F4")
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Power BI atualizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Power BI atualizado

----------------------------------------------------------------------------------------------------


# Resumo de Finalização do Processo

In [14]:
print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_8[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

     ✅  Processo finalizado

     ⏱️   Tempo de execução:

   0:00:24.477812

----------------------------------------------------------------------------------------------------
